# Create medium for models

In [14]:
import reframed
import cobra
import pandas as pd
import numpy as np
from os import listdir
import warnings
import json

In [2]:
import sys

sys.path.append("../functions/")
from minimal_medium import minimal_medium

In [3]:
%load_ext autoreload
%autoreload 2

### E. coli auxotrophs (Guillen et al. 2021) 

The medium used in the study was M9 medium, which is already defined.

### Gut microbiome (Venturelli et al. 2018) 

**Generate and load data**

In [4]:
with open('../data/Venturelli2018/generated/metname2metanetx.json') as json_file:
    metname2metanetx = json.load(json_file)

with open("../data/Venturelli2018/generated/metid2metname.json") as json_file:
    metid2metname = json.load(json_file)

dataset4 = pd.read_excel("../data/Venturelli2018/msb178157-sup-0005-datasetev4.xlsx",sheet_name="Dataset EV4",index_col=0)

media_db = pd.read_csv("https://raw.githubusercontent.com/cdanielmachado/carveme/refs/heads/master/carveme/data/input/media_db.tsv",sep="\t")
compounds = media_db[media_db.medium=="LB[-O2]"].compound.values

**Load models**

In [5]:
models = {}
with warnings.catch_warnings():
    warnings.filterwarnings('ignore')
    for path in listdir("../models/Venturelli2018/species_used/"):
        models[path[:-4]] = reframed.load_cbmodel("../models/Venturelli2018/species_used/"+path,exchange_detection="R_EX_(?!biomass__40__e__41__)")
    

**Create map between BiGG mets and AGORA exchange rxns**

In [6]:
mets_id_ag = []
mets_id_bigg = []
ex_rxns_ag = []
for model in models.values():
    for rxn in model.get_exchange_reactions():
        if "R_EX_" not in rxn:
            continue
        met_id_ag = model.reactions[rxn].get_substrates()[0]
        mets_id_ag.append(met_id_ag)
        met_id_bigg = model.metabolites[met_id_ag].\
                        metadata["XMLAnnotation"].\
                        split("bigg.metabolite/")[1].\
                        split("\"")[0]
        mets_id_bigg.append(met_id_bigg)
        ex_rxns_ag.append(rxn)
    
met2ex = dict(zip(mets_id_bigg,ex_rxns_ag))

In [7]:
for met in compounds:
    if met not in met2ex.keys():
        print(met)

amp
cmp
gmp
lipoate
mobd
ni2
ump


**Check growth on LB medium**

In [8]:
lb_env = reframed.Environment.from_reactions([met2ex[met] for met in compounds if met in met2ex.keys()])

In [9]:
for model_id,model in models.items():
    lb_env.apply(model,inplace=True,exclusive=True,warning=False)
    sol = reframed.FBA(model)
    print(f"{model_id}: {sol.fobj}")

Bacteroides_uniformis_ATCC_8492: 0.6167101596973709
Bacteroides_vulgatus_ATCC_8482: 0.0
Collinsella_aerofaciens_ATCC_25986: 0.0
Faecalibacterium_prausnitzii_A2_165: 0.0
Eggerthella_lenta_DSM_2243: 0.0
Prevotella_copri_CB7_DSM_18205: 0.5233877307380475
Clostridium_hiranonis_TO_931_DSM_13275: 0.0
Eubacterium_rectale_ATCC_33656: 0.9667459103245493
Desulfovibrio_piger_ATCC_29098: 0.0
Bacteroides_ovatus_ATCC_8483: 0.6948093215047774
Bacteroides_thetaiotaomicron_VPI_5482: 0.0
Blautia_hydrogenotrophica_DSM_10507: 0.0


**Defining a minimal media from LB where all members can grow**

In [10]:
# Compounds from study - these get priority (preferred reactions)
rxns = ["R_EX_"+met[2:].replace("91","40").replace("93","41") for met in metid2metname.keys() if "__e__" in met]
rxns.append("R_EX_malt__40__e__41__")
rxns.append("R_EX_cellb__40__e__41__")

missing_total = []
specific_solutions = []
sol_count_dfs = {}

# Find minimal medium for each model
for model_id,model in models.items():
    
    lb_env.apply(model,inplace=True,exclusive=True,warning=False)
    sol = reframed.FBA(model)
    print(f"{model_id}: {sol.fobj}")
    
    # If model already can grow - continue
    if sol.fobj>1e-6:
        continue
    
    # Find complete medium (based on reactions) for the model
    env_complete = reframed.Environment.complete(model)
    env_complete.apply(model,inplace=True,exclusive=True,warning=False)
    rxns_complete = env_complete.keys()
    # ...exclude specific reactions
    rxns_complete = list(set([e for e in rxns_complete if ("R_DM" not in e) and ("_acald_" not in e)]))
    rxns_complete = [rxn for rxn in rxns_complete if "_o2_" not in rxn]
    
    # Find minimal medium
    solutions = minimal_medium(model,exchange_reactions=list(rxns_complete),preferred_reactions=rxns,min_growth=0.1,n_solutions=1000,use_pool=True,min_mass_weight=True)
    
    print(solutions[1][0])
    
    # Count occurence of a metabolite in minimal medium (across 1000 solutions) 
    sol_count = dict()
    for sol in solutions[0]:
        for rxn in sol:
            if rxn in sol_count.keys():
                sol_count[rxn]+=1
            else:
                sol_count[rxn]=1
    sol_count_df = pd.Series(sol_count).sort_values(ascending=False)
    sol_count_dfs[model_id]=sol_count_df
    
    # Include the compounds that are present in >500 solutions in the extended medium (missing_total)
    missing = list((set(sol_count_df[sol_count_df>500].index)-set(lb_env.keys())))
    missing_total.extend(missing)

add2lb = list(set(missing_total))

Bacteroides_uniformis_ATCC_8492: 0.6167101596973709
Bacteroides_vulgatus_ATCC_8482: 0.0


/Users/idunmariaburgos/opt/anaconda3/envs/rcell_rec/lib/python3.8/site-packages/reframed/core/elements.py:62: UserWarning: Atomic weight not listed for elements: {'X'}
  warn(f"Atomic weight not listed for elements: {missing}")


Objective: 344424.11489917577
Status: Unknown

Collinsella_aerofaciens_ATCC_25986: 0.0
Objective: 166468.6285
Status: Unknown

Faecalibacterium_prausnitzii_A2_165: 0.0
Objective: 216376.57879999754
Status: Unknown

Eggerthella_lenta_DSM_2243: 0.0
Objective: 513388.9333
Status: Unknown

Prevotella_copri_CB7_DSM_18205: 0.5233877307380475
Clostridium_hiranonis_TO_931_DSM_13275: 0.0
Objective: 137464.2308
Status: Unknown

Eubacterium_rectale_ATCC_33656: 0.9667459103245493
Desulfovibrio_piger_ATCC_29098: 0.0
Objective: 102491.02029999999
Status: Unknown

Bacteroides_ovatus_ATCC_8483: 0.6948093215047774
Bacteroides_thetaiotaomicron_VPI_5482: 0.0
Objective: 212929.89836757528
Status: Unknown

Blautia_hydrogenotrophica_DSM_10507: 0.0
Objective: 196607.2274
Status: Unknown



**Summarize all results and test growth**

In [11]:
lb_extended = [met2ex[met] for met in compounds if met in met2ex.keys()]
lb_extended.extend(add2lb)
lb_extended.extend(rxns)

lb_extended = list(set(lb_extended))

env_lb_extend = reframed.Environment.from_reactions(lb_extended)

for model_id,model in models.items():
    env_lb_extend.apply(model,inplace=True,exclusive=True,warning=False)
    sol = reframed.pFBA(model)
    print(f"{model_id}: {sol.values['R_EX_biomass__40__e__41__']}")


Bacteroides_uniformis_ATCC_8492: 2.6734912428231907
Bacteroides_vulgatus_ATCC_8482: 1.7579309271402346
Collinsella_aerofaciens_ATCC_25986: 2.1195647297378217
Faecalibacterium_prausnitzii_A2_165: 2.2537638374451006
Eggerthella_lenta_DSM_2243: 0.39409295461247584
Prevotella_copri_CB7_DSM_18205: 2.325195428226256
Clostridium_hiranonis_TO_931_DSM_13275: 0.7601605197088038
Eubacterium_rectale_ATCC_33656: 3.5942509657612827
Desulfovibrio_piger_ATCC_29098: 0.48958790669265295
Bacteroides_ovatus_ATCC_8483: 3.8591009189796743
Bacteroides_thetaiotaomicron_VPI_5482: 2.493683496489884
Blautia_hydrogenotrophica_DSM_10507: 2.823920616242932


**Make a medium for SteadierCom**

In [12]:
lb_extended_copy = lb_extended.copy()
lb_extended_copy = [rxn.replace("40","91") for rxn in lb_extended_copy]
lb_extended_copy = [rxn.replace("41","93") for rxn in lb_extended_copy]
media_db_lb_extend = pd.DataFrame({"medium":["LB_extend" for rxn in lb_extended_copy],
              "compound":lb_extended_copy})

media_db_lb_extend

,medium,compound
0,LB_extend,R_EX_phe_L__91__e__93__
1,LB_extend,R_EX_h2o__91__e__93__
2,LB_extend,R_EX_cl__91__e__93__
3,LB_extend,R_EX_dad_2__91__e__93__
4,LB_extend,R_EX_val_L__91__e__93__
...,...,...
85,LB_extend,R_EX_cbl1__91__e__93__
86,LB_extend,R_EX_thr_L__91__e__93__
87,LB_extend,R_EX_cro4__91__e__93__
88,LB_extend,R_EX_ribflv__91__e__93__


In [13]:
media_db_lb_extend.to_csv("../data/Venturelli2018/generated/media_db_lb_extend.tsv",sep="\t",index=False)